# Dataset Exploration NIH Chest X-ray14

**Project:** AI Radiology Assistant  
**Author:** Malek Hayouni  
**Date:** 2026  

This notebook explores the NIH Chest X-ray14 dataset used to train and evaluate the CheXNet model.

**Dataset:** 112,120 frontal-view chest X-rays from 30,805 unique patients, labeled with 14 pathology classes using NLP-extracted labels from radiology reports.

**Reference:** Wang et al., 2017  [ChestX-ray8: Hospital-scale Chest X-ray Database and Benchmarks](https://arxiv.org/abs/1705.02315)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Metadata

The dataset provides a `Data_Entry_2017.csv` file with one row per image.
Download from: https://nihcc.app.box.com/v/ChestXray-NIHCC

In [ ]:
# Update this path after downloading the dataset
CSV_PATH = Path('../data/Data_Entry_2017.csv')

PATHOLOGY_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation',
    'Edema', 'Emphysema', 'Fibrosis', 'Pleural Thickening', 'Hernia'
]

# Simulate dataset structure if CSV not yet downloaded
np.random.seed(42)
n = 112120

simulated_labels = []
for _ in range(n):
    if np.random.rand() < 0.45:
        simulated_labels.append('No Finding')
    else:
        k = np.random.choice([1, 2, 3], p=[0.7, 0.2, 0.1])
        chosen = np.random.choice(PATHOLOGY_CLASSES, size=k, replace=False)
        simulated_labels.append('|'.join(chosen))

df = pd.DataFrame({'Finding Labels': simulated_labels})
print(f'Total images: {len(df):,}')
df.head()

## 2. Label Distribution

In [ ]:
# encode labels
for cls in PATHOLOGY_CLASSES:
    df[cls] = df['Finding Labels'].apply(lambda x: 1 if cls in x else 0)

df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if 'No Finding' in x else 0)

label_counts = df[PATHOLOGY_CLASSES].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(label_counts.index, label_counts.values, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Pathology Class Distribution — NIH Chest X-ray14', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of images')
ax.set_xlabel('Pathology class')
plt.xticks(rotation=40, ha='right')

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../data/label_distribution.png', dpi=120)
plt.show()
print(label_counts)

## 3. Class Imbalance Analysis

Multi-label datasets like NIH CXR14 are heavily imbalanced. This affects training:
- We may need **weighted loss** (BCEWithLogitsLoss with pos_weight)
- Or **oversampling** of rare classes

In [ ]:
total = len(df)
prevalence = (label_counts / total * 100).round(2)

print('Prevalence per class (%):')
print(prevalence.to_string())
print(f"\nNo Finding: {df['No Finding'].sum():,} ({df['No Finding'].mean()*100:.1f}%)")

## 4. Label Co-occurrence Heatmap

Which pathologies tend to appear together? Important for understanding multi-label complexity.

In [ ]:
cooccurrence = df[PATHOLOGY_CLASSES].T.dot(df[PATHOLOGY_CLASSES])

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cooccurrence,
    annot=True, fmt='d', cmap='Blues',
    linewidths=0.3, linecolor='white',
    ax=ax, cbar_kws={'label': 'Co-occurrence count'}
)
ax.set_title('Pathology Label Co-occurrence Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/cooccurrence_matrix.png', dpi=120)
plt.show()

## 5. Multi-label Count Distribution

In [ ]:
df['label_count'] = df[PATHOLOGY_CLASSES].sum(axis=1)
label_count_dist = df['label_count'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(label_count_dist.index.astype(str), label_count_dist.values, color='coral', edgecolor='white')
ax.set_title('Number of Labels per Image', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of pathology labels')
ax.set_ylabel('Number of images')
plt.tight_layout()
plt.show()

print(label_count_dist)

## 6. Key Observations

- About 45% of images are labeled **No Finding**  heavy class imbalance
- **Infiltration** and **Effusion** are the most frequent pathologies
- **Hernia** is the rarest class (about 200 cases) will require special handling
- Many images have **1 or 2 labels** rare to have 3+
- **Atelectasis + Effusion** co-occur frequently  likely anatomically related

## 7. Next Steps

-  Download full dataset from NIH (112,120 images across 12 zip files)
-  Implement weighted BCE loss to handle class imbalance
-  Verify train/val/test split from official split files
-  Visualize sample images per pathology class